# Lección 8 · Incertidumbre: Monte Carlo, simulación y escenarios

**Qué vas a aprender**
- Por qué planear con el **promedio** sale caro: el *flaw of averages*.
- **Escenarios** con probabilidad y el **valor esperado** de un plan.
- **Estocástica de 2 etapas**: decides hoy; mañana, en cada escenario, corriges con un **recurso** (*recourse*).
- Dos números para el jefe: cuánto vale saber el futuro (**EVPI**) y cuánto vale modelar la incertidumbre (**VSS**).
- **Monte Carlo**: generar miles de escenarios, resolver con una muestra (*SAA*) y evaluar el plan **fuera de la muestra**.
- **Simulación de eventos discretos**: colas en los muelles de una base. Intervalo de confianza de una simulación.
- El modelo crece: repuestos en las 3 bases de la lección 5, de 10 a 1.000 escenarios.
- Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Hoy no hay solver nuevo. La aleatoriedad sale de `numpy`, con **semilla fija**: cada vez que corras, salen los mismos números.

In [ ]:
%pip install -q pyomo highspy

In [ ]:
import functools
import io
import logging
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.repn import generate_standard_repn

SOLVER = pyo.SolverFactory("appsi_highs")
logging.getLogger("pyomo.contrib.appsi").setLevel(logging.ERROR)


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def tamano(m):
    # El tamaño del modelo: variables, restricciones y no ceros (coeficientes distintos de 0)
    variables = list(m.component_data_objects(pyo.Var))
    restricciones = list(m.component_data_objects(pyo.Constraint, active=True))
    return {"variables": len(variables),
            "binarias": sum(v.is_binary() for v in variables),
            "enteras": sum(v.is_integer() for v in variables),  # incluye las binarias
            "restricciones": len(restricciones),
            "no_ceros": sum(sum(c != 0 for c in generate_standard_repn(r.body).linear_coefs) for r in restricciones)}

## 1. Dónde quedamos
En la lección 7 todo era **dato**: la duración de cada proyecto, los días de mal tiempo, las multas. En la vida real, casi nada de eso se sabe de antemano.
Hoy los datos son **inciertos**: tienen varios valores posibles, cada uno con su probabilidad.

**Analogía.** La babcia y los pierogi de Wigilia. Tiene que amasar **antes** de saber cuántos llegan: ¿vienen los primos de Gdańsk o no?
- Si hace pocos y llegan todos, alguien sale corriendo a comprar al Żabka: caro y a última hora. Eso es el **recurso**.
- Si hace muchos y no llegan, sobran.
- Amasar es la **primera etapa** (se decide hoy). Correr al Żabka es la **segunda etapa** (se decide cuando ya sabes quién llegó).

La babcia no amasa para "el número promedio de invitados". Amasa pensando en **qué pasa si** llegan todos. Hoy aprendes a escribir eso.

## 2. Repuestos para el invierno
El parque eólico `parque_1` usa un **módulo convertidor** que falla con las tormentas. En octubre decides cuántos repuestos comprar para el invierno.

- Comprar ahora cuesta **40.000 €** por módulo.
- Si falla un módulo y no hay repuesto, se trae de urgencia: transporte aéreo, técnicos de fuera y días con la turbina parada. **250.000 €** por módulo.
- Lo que sobra se guarda para el año siguiente. Para hoy no cuenta.

Nadie sabe cómo será el invierno. Con los datos de años anteriores, el equipo arma 3 **escenarios**:

| Escenario | Probabilidad | Fallas |
|---|---|---|
| suave | 0,4 | 0 |
| normal | 0,4 | 2 |
| duro | 0,2 | 6 |

**Las 3 piezas:**
- **Decido:** cuántos módulos compro hoy. Y, cuando llega el invierno, cuántos traigo de urgencia.
- **Objetivo:** minimizar el costo **esperado**: el promedio de los costos de cada escenario, pesado por su probabilidad.
- **Restricciones:** en cada escenario, repuestos comprados + urgencias ≥ fallas.

In [ ]:
ESC = pd.DataFrame({"escenario": ["suave", "normal", "duro"], "prob": [0.4, 0.4, 0.2], "fallas": [0, 2, 6]})
COMPRA, URGENCIA = 40_000, 250_000

media = (ESC["prob"] * ESC["fallas"]).sum()
print("Fallas esperadas:", media)
ESC

### 2.1 Fuerza bruta: probar cada cantidad
Para cada cantidad $q$ de repuestos, el costo de un escenario es la compra más las urgencias que faltan.
El costo **esperado** pesa cada escenario por su probabilidad.

In [ ]:
def costo_esperado(q, esc=ESC, compra=COMPRA, urgencia=URGENCIA):
    falta = (esc["fallas"] - q).clip(lower=0)  # urgencias en cada escenario
    return compra * q + (esc["prob"] * urgencia * falta).sum()


tabla = pd.DataFrame({"compro": range(9)})
tabla["costo_esperado"] = [eur(costo_esperado(q)) for q in tabla["compro"]]
tabla

**Lectura:**
- Lo mejor es comprar **6 módulos**: **240.000 €** esperados. Con 7 ya sobra uno seguro: sube 40.000 €.
- Comprar el promedio (2) cuesta **280.000 €** esperados.
- Entre 2 y 6 el costo baja 10.000 € por cada módulo. Por qué: cada módulo extra cuesta 40.000 € y se usa solo en el invierno duro (20 %).
  Ahorra $0{,}2 \cdot 250.000 = 50.000$ € esperados. Gana 10.000 €.

**La regla del módulo número k.** Vale la pena si $P(\text{fallas} \ge k) \cdot 250.000 > 40.000$, o sea si $P(\text{fallas} \ge k) > 16\,\%$.
Para $k = 3, \dots, 6$, esa probabilidad es 20 %: se compran. Para $k = 7$, es 0. Es el **vendedor de periódicos** (*newsvendor*), el problema más viejo de inventarios.

### 2.2 El error del promedio
¿Qué pasa si planeas con el promedio, como si el invierno fuera seguro? Un solo escenario: 2 fallas.

In [ ]:
plan_promedio = 2
print("Lo que promete el plan con el promedio:", eur(COMPRA * plan_promedio), "(compra 2, falla 2, cero urgencias)")
print("Lo que cuesta en el invierno real:     ", eur(costo_esperado(plan_promedio)))
print("Lo que cuesta el mejor plan:           ", eur(costo_esperado(6)))

**Lectura:**
- El plan con el promedio **promete 80.000 €** y cuesta **280.000 €** esperados. Se equivoca en 3,5 veces.
- Es el ***flaw of averages***: el costo del plan con el promedio no es el promedio de los costos.
  El costo es asimétrico: que falte uno cuesta 250.000 €; que sobre uno, 40.000 €. El promedio esconde ese lado caro.

### En notación
Conjuntos: $S$ = escenarios. Datos: $\pi_s$ = probabilidad del escenario $s$, $D_s$ = fallas en $s$, $c$ = compra, $g$ = urgencia.
Variables: $q$ = repuestos que compro hoy (**primera etapa**), $e_s$ = urgencias en el escenario $s$ (**segunda etapa**, recurso).

$$
\begin{aligned}
\min \quad & c\, q + \sum_{s \in S} \pi_s\, g\, e_s \\
\text{s.a.} \quad & q + e_s \ge D_s && \forall\, s \in S && \text{(cubrir las fallas)} \\
& q \in \mathbb{Z}_{\ge 0}, \quad e_s \ge 0 && \forall\, s \in S
\end{aligned}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $S$, $s$ | "ese", "escenario ese" | los futuros posibles | `ESC["escenario"]` |
| $\pi_s$ | "pi de ese" | probabilidad del escenario; $\sum_s \pi_s = 1$ | `ESC["prob"]` |
| $\sum_{s} \pi_s\, g\, e_s$ | "suma de pi por ge por e" | el **valor esperado** del costo de urgencias: $\mathbb{E}[g\, e]$ | `(prob * g * e).sum()` |
| $\mathbb{E}[\cdot]$ | "esperanza de…" | promedio pesado por la probabilidad | `np.average(x, weights=prob)` |
| $q$ sin índice $s$ | "cu" | **una** decisión para todos los escenarios: se toma antes de saber | `m.q` |
| $e_s$ con índice $s$ | "e de ese" | una decisión **por** escenario: se toma después de saber | `m.e[s]` |
| $\mathbb{Z}_{\ge 0}$ | "enteros no negativos" | 0, 1, 2, … | `pyo.NonNegativeIntegers` |

**La clave está en los índices.** $q$ no lleva $s$: es la misma en los 3 inviernos. $e_s$ sí lleva $s$: cambia con el invierno.
Esto se llama ***non-anticipativity***: no puedes decidir hoy con información de mañana.

In [ ]:
def dos_etapas(esc, compra=COMPRA, urgencia=URGENCIA):
    S = list(esc["escenario"])
    prob = dict(zip(S, esc["prob"]))
    D = dict(zip(S, esc["fallas"]))
    m = pyo.ConcreteModel()
    m.S = pyo.Set(initialize=S)
    m.q = pyo.Var(within=pyo.NonNegativeIntegers)  # primera etapa: sin índice s
    m.e = pyo.Var(m.S, within=pyo.NonNegativeReals)  # segunda etapa: una por escenario
    m.obj = pyo.Objective(expr=compra * m.q + sum(prob[s] * urgencia * m.e[s] for s in m.S), sense=pyo.minimize)
    m.cubre = pyo.Constraint(m.S, rule=lambda m, s: m.q + m.e[s] >= D[s])
    SOLVER.solve(m)
    return m


m2 = dos_etapas(ESC)
print(f"Compro {pyo.value(m2.q):.0f} módulos | costo esperado {eur(pyo.value(m2.obj))}")
print("Urgencias por escenario:", {s: round(pyo.value(m2.e[s])) for s in m2.S})
print("Tamaño:", tamano(m2))

**Lectura:**
- El modelo llega a lo mismo que la fuerza bruta: **6 módulos, 240.000 €**. Cero urgencias en los 3 escenarios.
- Esta forma, con una copia de la segunda etapa por escenario, se llama **forma extendida** (*extensive form*) o *equivalente determinista*.

**Los dos tamaños.**
- Modelo: $1 + |S|$ variables y $|S|$ restricciones. Crece **lineal** con los escenarios. Con 3 escenarios, 4 variables.
- Espacio de búsqueda: $q$ va de 0 a 6. Son 7 planes. El espacio es chico; lo difícil es **calcular el costo** de cada plan cuando hay miles de escenarios (sección 3).

### 2.3 Dos números para el jefe: EVPI y VSS
- **Esperar y ver** (*wait-and-see*, WS): si supieras el invierno antes de comprar, comprarías justo las fallas. Es el costo con un pronóstico perfecto.
- **Aquí y ahora** (*here-and-now*, RP): el plan de 2 etapas. Una sola compra para todos los inviernos.
- **Plan con el promedio** (EEV): el costo esperado real del plan que usó el promedio.

$$
\begin{aligned}
\text{EVPI} &= \text{RP} - \text{WS} && \text{(cuánto vale saber el futuro)} \\
\text{VSS} &= \text{EEV} - \text{RP} && \text{(cuánto vale modelar la incertidumbre)}
\end{aligned}
$$

In [ ]:
WS = sum(r.prob * COMPRA * r.fallas for r in ESC.itertuples())  # en cada escenario compras justo lo que falla
RP = pyo.value(m2.obj)
EEV = costo_esperado(plan_promedio)
print("WS  (pronóstico perfecto):", eur(WS))
print("RP  (2 etapas):           ", eur(RP))
print("EEV (plan con el promedio):", eur(EEV))
print("EVPI = RP - WS  =", eur(RP - WS))
print("VSS  = EEV - RP =", eur(EEV - RP))

**Lectura:**
- **EVPI = 160.000 €** (*expected value of perfect information*). Es lo máximo que pagarías por saber el invierno de antemano.
  Un mejor pronóstico meteorológico, o un modelo de fallas mejor, vale como mucho eso. Es el puente con la lección 11.
- **VSS = 40.000 €** (*value of the stochastic solution*). Es lo que ganas por modelar los 3 escenarios en vez de usar el promedio.
- Siempre se cumple $\text{WS} \le \text{RP} \le \text{EEV}$. Si el VSS sale casi 0, el promedio basta y no hace falta el modelo estocástico.

## 3. Monte Carlo: miles de escenarios
Tres escenarios son una caricatura. En la realidad, las fallas de un invierno son un número al azar.
El equipo de datos te da un **modelo generador**:

1. Se sortea el tipo de invierno: suave (50 %), normal (35 %) o duro (15 %).
2. Según el tipo, las fallas siguen una **Poisson** con media $\lambda$ = 1, 3 o 7.

**Monte Carlo** = repetir el sorteo muchas veces y mirar los resultados. Cada sorteo es un escenario.
El nombre viene del casino: cada escenario es una tirada de la ruleta.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $D \sim \text{Poisson}(\lambda)$ | "D sigue una Poisson de lambda" | conteo de eventos raros con media $\lambda$: 0, 1, 2, … | `rng.poisson(lam)` |
| $\lambda$ | "lambda" | fallas esperadas en ese tipo de invierno | `lam` |
| $\hat{\theta}$ | "theta gorro" | una **estimación** de $\theta$ hecha con una muestra | `muestra.mean()` |

In [ ]:
INVIERNOS = pd.DataFrame({"invierno": ["suave", "normal", "duro"], "prob": [0.5, 0.35, 0.15], "lam": [1.0, 3.0, 7.0]})


def muestrear(n, semilla):
    rng = np.random.default_rng(semilla)
    tipo = rng.choice(len(INVIERNOS), size=n, p=INVIERNOS["prob"])
    return pd.DataFrame({"escenario": [f"s{i}" for i in range(n)], "prob": 1 / n,
                         "invierno": INVIERNOS["invierno"].values[tipo],
                         "fallas": rng.poisson(INVIERNOS["lam"].values[tipo])})


muestrear(1000, 0).head()

Para medir **el costo real** de un plan, sorteamos 200.000 inviernos una sola vez. Es nuestro "mundo real".
Con $q$ fijo, el costo de cada invierno es una cuenta: no hace falta solver.

In [ ]:
MUNDO = muestrear(200_000, semilla=999)


def costo_real(q, mundo=MUNDO):
    return COMPRA * q + URGENCIA * np.maximum(mundo["fallas"].to_numpy() - q, 0).mean()


print(f"Fallas por invierno: media {MUNDO['fallas'].mean():.2f}, máximo {MUNDO['fallas'].max()}")
print("P(fallas >= k):", {k: round(float((MUNDO["fallas"] >= k).mean()), 3) for k in range(1, 11)})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
conteo = MUNDO["fallas"].value_counts(normalize=True).sort_index()
ax.bar(conteo.index, conteo.values, color="#60a5fa", edgecolor="black")
ax.axvline(MUNDO["fallas"].mean(), color="#dc2626", ls="--", label=f"media = {MUNDO['fallas'].mean():.1f}")
ax.set(xlabel="fallas en el invierno", ylabel="probabilidad", title="200.000 inviernos simulados", xlim=(-0.7, 18))
ax.legend()
plt.tight_layout()
plt.show()

**Lectura:**
- La media es **2,6 fallas**, pero la cola es larga: 1 de cada 7 inviernos tiene 6 o más. El peor tiene 21.
- La regla del módulo k: comprar mientras $P(\text{fallas} \ge k) > 16\,\%$. Aquí $P(\ge 5) = 0{,}19$ y $P(\ge 6) = 0{,}14$: se compran **5**.

### 3.1 SAA: resolver con una muestra
**SAA** (*sample average approximation*): sorteas $N$ escenarios, cada uno con probabilidad $1/N$, y resuelves la forma extendida.
Es el mismo modelo de la sección 2, con más filas.

$$
\min_{q} \; c\, q + \frac{1}{N} \sum_{s=1}^{N} g\, \max\{0,\ D_s - q\}
$$

Preguntas: ¿cuántos escenarios hacen falta? ¿Cuánto se equivoca el modelo sobre su propio costo?
Para cada $N$, 5 muestras distintas. Para cada plan, dos costos: el que **promete** (en su muestra) y el **real** (en el mundo de 200.000).

In [ ]:
filas = []
for n in [10, 100, 1000, 10_000]:
    for rep in range(5):
        muestra = muestrear(n, semilla=100 * n + rep)
        inicio = time.perf_counter()
        mm = dos_etapas(muestra)
        q = round(pyo.value(mm.q))
        filas.append({"N": n, "muestra": rep, "compro": q, "promete": round(pyo.value(mm.obj)),
                      "real": round(costo_real(q)), "segundos": round(time.perf_counter() - inicio, 3)})
saa = pd.DataFrame(filas)
saa

In [ ]:
resumen_saa = saa.groupby("N").agg(compras=("compro", lambda s: sorted(set(s))), promete_medio=("promete", "mean"),
                                   real_medio=("real", "mean"), segundos=("segundos", "mean"))
resumen_saa["optimismo_%"] = (100 * (resumen_saa["real_medio"] - resumen_saa["promete_medio"]) / resumen_saa["real_medio"]).round(1)
resumen_saa.round(0)

**Lectura:**
- Con **10 escenarios**, las 5 muestras dan planes distintos: de 1 a 4 módulos. El plan depende de la suerte de la muestra.
- Con 10 escenarios el modelo **promete** unos 160.000 € y el plan cuesta unos 360.000 € en el mundo. Es **optimista**: elige el plan que mejor le va a **su** muestra.
  Es el sobreajuste (*overfitting*) del ML, en optimización.
- Desde **1.000 escenarios** las 5 muestras dan lo mismo: **5 módulos**, unos 298.000 €. La promesa queda a menos de 1 % del costo real.
- El tiempo crece lineal con $N$: el modelo tiene $1 + N$ variables.

**La regla práctica:** se resuelve con $N$ escenarios y **se evalúa con otros**, muchos más. Como train y test.
Se sube $N$ hasta que el plan deje de cambiar.

### 3.2 ¿Qué tan seguro es el costo? Intervalo de confianza
Cada estimación de Monte Carlo es un promedio. Su error baja con $\sqrt{N}$: para la mitad de error, 4 veces más escenarios.

$$
\bar{x} \pm 1{,}96 \cdot \frac{s}{\sqrt{N}}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\bar{x}$ | "equis barra" | promedio de la muestra | `x.mean()` |
| $s$ | "ese" | desviación estándar de la muestra | `x.std(ddof=1)` |
| $s / \sqrt{N}$ | "error estándar" | cuánto se mueve el promedio de una muestra a otra | `x.std(ddof=1) / np.sqrt(len(x))` |
| $1{,}96$ | | 95 % de una normal cae a ±1,96 errores estándar | `scipy.stats.norm.ppf(0.975)` |

In [ ]:
def intervalo(x):
    x = np.asarray(x, dtype=float)
    se = x.std(ddof=1) / np.sqrt(len(x))
    return x.mean(), x.mean() - 1.96 * se, x.mean() + 1.96 * se


for n in [100, 1000, 10_000, 100_000]:
    falla = MUNDO["fallas"].to_numpy()[:n]
    costos = COMPRA * 5 + URGENCIA * np.maximum(falla - 5, 0)  # costo de cada invierno con 5 módulos
    media_n, bajo, alto = intervalo(costos)
    print(f"N = {n:>7}: {eur(media_n):>11}  IC 95 % [{eur(bajo)}; {eur(alto)}]  ancho {eur(alto - bajo)}")

**Lectura:**
- Con 100 inviernos, el intervalo del plan de 5 módulos mide unos **130.000 €**: no sirve para comparar planes que se separan por 10.000 €.
- Con 100.000 inviernos baja a unos **4.000 €**. Cada ×10 en $N$ divide el ancho por $\sqrt{10} \approx 3{,}2$.
- Cuando compares dos planes, mira si sus intervalos se tocan. Si se tocan, aún no sabes cuál es mejor (ejercicio 2).

## 4. Simulación de eventos discretos: los muelles de la base
Hay problemas donde la incertidumbre se **acumula en el tiempo**: las colas.

La base de Łeba tiene muelles para cargar los barcos de suministro. Los barcos llegan al azar, uno cada 3 h en promedio.
Cargar toma 5 h en promedio (desviación 1,5 h). ¿Cuántos muelles hacen falta para que la espera media sea menor a 2 h?

**La cuenta con promedios:** cada barco ocupa 5 h y llega uno cada 3 h. Con 2 muelles, cada uno trabaja $5 / (3 \cdot 2) = 83\,\%$ del tiempo. Parece que alcanza.

**Analogía.** Las cajas del Biedronka un sábado. En promedio alcanzan, pero la gente llega en grupos y se forman filas.

**Simulación de eventos discretos.** El reloj salta de evento en evento (llega un barco, termina una carga). Cada barco que llega toma el muelle que se libera primero.
Un mes simulado es **una réplica**. Se repite 200 veces con semillas distintas.

In [ ]:
LLEGADA_MEDIA, CARGA_MEDIA, CARGA_SD, HORAS = 3.0, 5.0, 1.5, 24 * 30


def un_mes(muelles, semilla):
    rng = np.random.default_rng(semilla)
    llegadas = np.cumsum(rng.exponential(LLEGADA_MEDIA, size=400))
    llegadas = llegadas[llegadas < HORAS]
    cargas = np.maximum(0.5, rng.normal(CARGA_MEDIA, CARGA_SD, size=len(llegadas)))
    libre = np.zeros(muelles)  # hora en que se libera cada muelle
    esperas = np.empty(len(llegadas))
    for i, (llega, carga) in enumerate(zip(llegadas, cargas)):
        k = libre.argmin()  # el muelle que se libera primero
        empieza = max(llega, libre[k])
        esperas[i] = empieza - llega
        libre[k] = empieza + carga
    return esperas.mean()


print(f"Uso con 2 muelles según la cuenta con promedios: {CARGA_MEDIA / LLEGADA_MEDIA / 2:.0%}")
print(f"Un mes con 2 muelles: espera media {un_mes(2, semilla=0):.1f} h")

In [ ]:
REPLICAS = 200
replicas = pd.DataFrame({k: [un_mes(k, s) for s in range(REPLICAS)] for k in [2, 3, 4]})
filas = []
for k in replicas:
    media_k, bajo, alto = intervalo(replicas[k])
    filas.append({"muelles": k, "espera_media_h": round(media_k, 2), "IC_95": f"[{bajo:.2f}, {alto:.2f}]",
                  "P(mes > 2 h)": f"{(replicas[k] > 2).mean():.0%}", "peor_mes_h": round(replicas[k].max(), 1)})
pd.DataFrame(filas)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for k, color in zip([2, 3], ["#f97316", "#60a5fa"]):
    ax.hist(replicas[k], bins=np.linspace(0, 21, 64), alpha=0.7, color=color, edgecolor="black", label=f"{k} muelles")
ax.axvline(2, color="#dc2626", ls="--", label="meta: 2 h")
ax.set(xlabel="espera media del mes (h)", ylabel="meses", title="200 meses simulados")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura:**
- Con 2 muelles la espera media es **5,6 h**, casi 3 veces la meta. **Todos** los meses simulados pasan de 2 h, y el peor llega a 20 h. La cuenta del 83 % no ve las colas.
- Un solo mes no basta para decidir: el de la semilla 0 dio 2,1 h. Por eso se hacen 200 réplicas y se reporta el intervalo.
- Con 3 muelles baja a **0,6 h** y ningún mes pasa de 2 h. Con 4, a 0,1 h: el cuarto muelle casi no aporta.
- La recomendación es **3 muelles**.

**Simulación contra optimización.** La simulación **evalúa** un plan (¿qué pasa con 3 muelles?). La optimización **elige** entre planes.
Juntas: la optimización propone y la simulación revisa con toda la variabilidad. Eso es un ***digital twin*** sencillo.
Para sistemas más grandes (varios recursos, prioridades, turnos) se usa una librería como **SimPy**: la lógica es la misma.

## 5. El modelo crece: repuestos en las 3 bases
Volvemos a la red de la lección 5: bases en Władysławowo, Łeba y Ustka, y 4 parques eólicos.
Ahora el módulo convertidor se guarda en las bases. Cuando falla uno en un parque, sale un barco desde una base con stock.

- **Primera etapa (octubre):** en qué bases habilito bodega para el módulo (20.000 € por base) y cuántos módulos guardo en cada una (40.000 € cada uno, máximo 6 por base).
- **Segunda etapa (cada invierno):** desde qué base atiendo cada falla. El envío cuesta más mientras más lejos (barco y días de turbina parada). Si no hay stock, urgencia a 250.000 €.
- **Lo nuevo:** las tormentas golpean a **todos los parques a la vez**. Un invierno duro es duro en los 4. Las fallas están **correlacionadas**.

**Las 3 piezas:**
- **Decido:** bases habilitadas y stock por base (hoy); envíos y urgencias en cada escenario (mañana).
- **Objetivo:** costo de bases + stock + costo esperado de envíos y urgencias.
- **Restricciones:** stock solo en bases habilitadas; en cada escenario, no envío más de lo que hay; cada falla se atiende con envío o urgencia.

Tres tablas, como en SQL. Costos en euros.

In [ ]:
bases = pd.read_csv(io.StringIO('''base,costo_habilitar,capacidad
Wladyslawowo,20000,6
Leba,20000,6
Ustka,20000,6
'''))
parques = pd.read_csv(io.StringIO('''parque,fallas_normal
parque_1,0.8
parque_2,1.2
parque_3,1.0
parque_4,0.6
'''))
envio = pd.read_csv(io.StringIO('''base,parque,costo_envio
Wladyslawowo,parque_1,5000
Wladyslawowo,parque_2,30000
Wladyslawowo,parque_3,60000
Wladyslawowo,parque_4,90000
Leba,parque_1,40000
Leba,parque_2,5000
Leba,parque_3,30000
Leba,parque_4,60000
Ustka,parque_1,90000
Ustka,parque_2,50000
Ustka,parque_3,5000
Ustka,parque_4,5000
'''))
envio.pivot(index="base", columns="parque", values="costo_envio")

Los escenarios, en **formato largo**: una fila por (escenario, parque). El tipo de invierno se sortea **una vez por escenario** y vale para los 4 parques.
Las fallas de un parque siguen una Poisson con media `fallas_normal × factor`: el factor es 0,4 en invierno suave, 1 en normal y 2,5 en duro.

In [ ]:
FACTOR = pd.DataFrame({"invierno": ["suave", "normal", "duro"], "prob": [0.5, 0.35, 0.15], "factor": [0.4, 1.0, 2.5]})


def escenarios_red(n, semilla, correlado=True):
    rng = np.random.default_rng(semilla)
    n_parques = len(parques)
    if correlado:  # un invierno por escenario, el mismo para todos los parques
        tipo = np.repeat(rng.choice(3, size=n, p=FACTOR["prob"])[:, None], n_parques, axis=1)
    else:  # cada parque con su propio invierno (lo usas en el ejercicio 3)
        tipo = rng.choice(3, size=(n, n_parques), p=FACTOR["prob"])
    fallas = rng.poisson(FACTOR["factor"].to_numpy()[tipo] * parques["fallas_normal"].to_numpy()[None, :])
    ancho = pd.DataFrame(fallas, columns=parques["parque"]).assign(escenario=[f"s{i:04d}" for i in range(n)],
                                                                   invierno=FACTOR["invierno"].to_numpy()[tipo[:, 0]])
    return ancho.melt(id_vars=["escenario", "invierno"], var_name="parque", value_name="fallas")


esc200 = escenarios_red(200, semilla=200)
print("Filas:", len(esc200), "| escenarios:", esc200["escenario"].nunique())
esc200.head()

### En notación
Conjuntos: $B$ = bases, $F$ = parques, $S$ = escenarios (cada uno con $\pi_s = 1/N$).
Datos: $h_b$ = costo de habilitar, $u_b$ = capacidad, $c$ = compra, $k_{bf}$ = costo de envío, $g$ = urgencia, $D_{fs}$ = fallas del parque $f$ en el escenario $s$.
Variables de **primera etapa**: $z_b \in \{0,1\}$ (habilito la base), $y_b \in \mathbb{Z}_{\ge 0}$ (stock).
Variables de **segunda etapa**: $x_{bfs} \ge 0$ (módulos de $b$ a $f$ en $s$), $e_{fs} \ge 0$ (urgencias).

$$
\begin{aligned}
\min \quad & \sum_{b \in B} \big( h_b\, z_b + c\, y_b \big) + \sum_{s \in S} \pi_s \Big( \sum_{b \in B} \sum_{f \in F} k_{bf}\, x_{bfs} + \sum_{f \in F} g\, e_{fs} \Big) \\
\text{s.a.} \quad & y_b \le u_b\, z_b && \forall\, b \in B && \text{(stock solo en bases habilitadas)} \\
& \sum_{f \in F} x_{bfs} \le y_b && \forall\, b \in B,\ s \in S && \text{(no envío lo que no tengo)} \\
& \sum_{b \in B} x_{bfs} + e_{fs} = D_{fs} && \forall\, f \in F,\ s \in S && \text{(cada falla se atiende)} \\
& z_b \in \{0, 1\}, \quad y_b \in \mathbb{Z}_{\ge 0}, \quad x_{bfs} \ge 0, \quad e_{fs} \ge 0
\end{aligned}
$$

Fíjate en los índices otra vez: $z_b$ y $y_b$ no llevan $s$. Todo lo que lleva $s$ es segunda etapa.
La segunda etapa de cada escenario es un **transporte** de la lección 5: las bases son la oferta ($y_b$) y los parques la demanda ($D_{fs}$).
El modelo estocástico es **$N$ copias del transporte**, unidas por el stock de la primera etapa.

In [ ]:
def red_dos_etapas(esc, fijar=None, relajar=False):
    # fijar: {base: stock} para evaluar un plan dado (solo se resuelve la segunda etapa)
    S = sorted(esc["escenario"].unique())
    prob = 1 / len(S)
    B, F = list(bases["base"]), list(parques["parque"])
    D = esc.set_index(["escenario", "parque"])["fallas"].to_dict()
    k = envio.set_index(["base", "parque"])["costo_envio"].to_dict()
    h = bases.set_index("base")["costo_habilitar"].to_dict()
    u = bases.set_index("base")["capacidad"].to_dict()

    m = pyo.ConcreteModel()
    m.B, m.F, m.S = pyo.Set(initialize=B), pyo.Set(initialize=F), pyo.Set(initialize=S)
    m.z = pyo.Var(m.B, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.y = pyo.Var(m.B, within=pyo.NonNegativeReals if relajar else pyo.NonNegativeIntegers)
    m.x = pyo.Var(m.B, m.F, m.S, within=pyo.NonNegativeReals)
    m.e = pyo.Var(m.F, m.S, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(h[b] * m.z[b] + COMPRA * m.y[b] for b in B)
                          + prob * sum(k[b, f] * m.x[b, f, s] for b in B for f in F for s in S)
                          + prob * URGENCIA * sum(m.e[f, s] for f in F for s in S), sense=pyo.minimize)
    m.capacidad = pyo.Constraint(m.B, rule=lambda m, b: m.y[b] <= u[b] * m.z[b])
    m.stock = pyo.Constraint(m.B, m.S, rule=lambda m, b, s: sum(m.x[b, f, s] for f in F) <= m.y[b])
    m.atiende = pyo.Constraint(m.F, m.S, rule=lambda m, f, s: sum(m.x[b, f, s] for b in B) + m.e[f, s] == D[s, f])
    if fijar is not None:
        for b in B:
            m.y[b].fix(fijar[b])
            m.z[b].fix(int(fijar[b] > 0))

    inicio = time.perf_counter()
    SOLVER.solve(m)
    m.segundos = time.perf_counter() - inicio
    return m


def stock_de(m):
    return {b: round(pyo.value(m.y[b])) for b in m.B}


mr = red_dos_etapas(esc200)
print("Stock por base:", stock_de(mr))
print("Costo esperado:", eur(pyo.value(mr.obj)), f"| {mr.segundos:.2f} s")
print("Tamaño:", tamano(mr))

### 5.1 La solución, de vuelta a tablas
Con el plan de primera etapa fijo, cada escenario tiene sus envíos. Los pasamos a una tabla larga y la resumimos con `groupby`, como en SQL.

In [ ]:
envios = pd.DataFrame([(b, f, s, pyo.value(mr.x[b, f, s])) for b, f, s in mr.x if pyo.value(mr.x[b, f, s]) > 1e-6],
                      columns=["base", "parque", "escenario", "modulos"])
urgencias = pd.DataFrame([(f, s, pyo.value(mr.e[f, s])) for f, s in mr.e if pyo.value(mr.e[f, s]) > 1e-6],
                         columns=["parque", "escenario", "urgencias"])
N200 = esc200["escenario"].nunique()
por_parque = (esc200.groupby("parque")["fallas"].sum().div(N200).rename("fallas_esperadas").to_frame()
              .join(envios.groupby("parque")["modulos"].sum().div(N200).rename("desde_base"))
              .join(urgencias.groupby("parque")["urgencias"].sum().div(N200).rename("urgencias"))
              .fillna(0).round(2))
por_parque

In [ ]:
# De qué base sale cada parque, en promedio por invierno
envios.pivot_table(index="base", columns="parque", values="modulos", aggfunc="sum").div(N200).fillna(0).round(2)

In [ ]:
# Top 5: los inviernos más caros con este plan
costo_esc = (esc200.groupby(["escenario", "invierno"])["fallas"].sum().rename("fallas").reset_index()
             .merge(urgencias.groupby("escenario")["urgencias"].sum().reset_index(), how="left").fillna({"urgencias": 0}))
costo_esc["costo_urgencias"] = (costo_esc["urgencias"] * URGENCIA).map(eur)
costo_esc.nlargest(5, "urgencias")

In [ ]:
# ¿Cuántos inviernos de cada tipo terminan con urgencias?
costo_esc.assign(con_urgencias=costo_esc["urgencias"] > 0).groupby("invierno").agg(
    escenarios=("escenario", "count"), fallas_medias=("fallas", "mean"), con_urgencias=("con_urgencias", "mean")).round(2)

**Lectura:**
- El plan habilita **Łeba y Ustka** con **3 módulos cada una**. Władysławowo queda sin bodega: `parque_1` se atiende desde Łeba (40.000 € el envío).
- Ustka cubre `parque_3` y `parque_4` (5.000 € cada envío). Łeba cubre `parque_2` y lo que sobre.
- Las urgencias se concentran en `parque_1` (0,27 por invierno): cuando falta stock, el modelo lo usa en los parques con envío barato y deja de urgencia al que está lejos de Łeba.
- El riesgo está en los inviernos **duros**: el 71 % termina con urgencias. De los normales, el 6 %; de los suaves, el 1 %.
  Las tormentas golpean a los 4 parques a la vez y el stock de 6 no alcanza.
- El costo del plan es el promedio de inviernos muy distintos. Los peores, con 12 a 18 fallas, cuestan entre 1,5 y 3 millones en urgencias.

### 5.2 El modelo crece: de 10 a 1.000 escenarios
Para cada $N$: resolvemos, y evaluamos el plan en **5.000 escenarios nuevos** (fuera de la muestra), con la primera etapa fija.
Agregamos el plan **con el promedio**: un único escenario con las fallas esperadas de cada parque.
Esta celda tarda alrededor de 40 segundos.

In [ ]:
EVAL = escenarios_red(5000, semilla=12345)


@functools.lru_cache(maxsize=None)
def costo_fuera(stock):  # stock: tupla ((base, módulos), ...); el cache evita evaluar dos veces el mismo plan
    return pyo.value(red_dos_etapas(EVAL, fijar=dict(stock)).obj)


filas = []
for n in [10, 50, 200, 1000]:
    mn = red_dos_etapas(escenarios_red(n, semilla=n))
    t = tamano(mn)
    filas.append({"escenarios": n, "variables": t["variables"], "restricciones": t["restricciones"], "no_ceros": t["no_ceros"],
                  "stock": stock_de(mn), "promete": eur(pyo.value(mn.obj)),
                  "real": eur(costo_fuera(tuple(stock_de(mn).items()))), "segundos": round(mn.segundos, 2)})

fallas_esperadas = (FACTOR["prob"] * FACTOR["factor"]).sum() * parques["fallas_normal"]
esc_media = parques.assign(escenario="promedio", fallas=fallas_esperadas)[["escenario", "parque", "fallas"]]
mm = red_dos_etapas(esc_media)
filas.append({"escenarios": "promedio", "variables": tamano(mm)["variables"], "restricciones": tamano(mm)["restricciones"],
              "no_ceros": tamano(mm)["no_ceros"], "stock": stock_de(mm), "promete": eur(pyo.value(mm.obj)),
              "real": eur(costo_fuera(tuple(stock_de(mm).items()))), "segundos": round(mm.segundos, 2)})
escala = pd.DataFrame(filas)
escala

**Cómo se lee:**
- El modelo crece **lineal** con los escenarios: con 1.000 tiene 16.006 variables y 7.003 restricciones. Las enteras siguen siendo 6.
- Desde **200 escenarios** el plan se estabiliza: 3 módulos en Łeba y 3 en Ustka. Su costo real es unos **446.000 €** por invierno.
- Con 10 escenarios el plan es otro (5 módulos en Władysławowo y 2 en Ustka) y cuesta unos 465.000 €. Con 50, guarda uno menos en Ustka.
- El plan **con el promedio** guarda solo 4 módulos (1 + 1 + 2), en las 3 bases. Promete unos 240.000 € y cuesta unos **480.000 €**: el doble de lo prometido.
  Su VSS: unos 34.000 € por invierno. Y el error al presupuestar, más de 200.000 €.

**Los dos tamaños.**
- Modelo: $2|B| + N(|B||F| + |F|)$ variables. Lineal en $N$.
- Espacio de búsqueda de la primera etapa: cada base guarda de 0 a 6 módulos, $7^{|B|} = 343$ planes. **Chico.**
- Lo difícil no es elegir entre 343 planes: es **calcular el costo de cada uno**, un valor esperado sobre infinitos inviernos.
  Por eso el modelo crece con $N$ y no con el espacio de búsqueda. Con muchas bases o muchos ítems, se usa **descomposición de Benders** (*L-shaped*): resuelve cada escenario aparte y le pasa cortes a la primera etapa.

**Lectura para el jefe de logística:**
- Guardar **6 módulos convertidores**: 3 en Łeba y 3 en Ustka. Władysławowo no necesita bodega para este ítem.
- Costo esperado: **unos 446.000 € por invierno**. Planear con el invierno promedio daría un presupuesto de 240.000 € que no se cumple: el costo real de ese plan es unos 480.000 €.
- El riesgo está en los **inviernos duros** (15 %): casi todas las urgencias salen de ahí. Si se quiere bajar ese riesgo, la palanca es stock extra **antes** de un pronóstico de invierno duro, no más bodegas.

## 6. ¿Qué herramienta para la incertidumbre?
Una sola regla para empezar:

> **Primero evalúa (simulación o Monte Carlo) el plan que tienes. Si el costo de la incertidumbre es grande y hay decisiones que se corrigen después, pasa a un modelo de 2 etapas.**

| Situación | Herramienta | En la lección |
|---|---|---|
| Un dato incierto, quiero ver si el plan cambia | sensibilidad por re-solve | lección 3 |
| Tengo un plan; quiero su costo y su riesgo con toda la variabilidad | Monte Carlo o simulación | secciones 3 y 4 |
| Hay colas, esperas, recursos que se liberan en el tiempo | simulación de eventos discretos | muelles |
| Decido hoy, corrijo mañana cuando sé qué pasó | estocástica de 2 etapas (SAA) | repuestos |
| Debe funcionar en **todo** un rango, sin probabilidades | optimización robusta | — |
| Se puede fallar, pero rara vez ("95 % de los inviernos sin urgencias") | restricción probabilística (*chance constraint*) | — |
| Me importa el peor 5 % de los casos, no solo el promedio | CVaR en el objetivo | — |

## 7. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Escenario | Un futuro posible, con su probabilidad | invierno suave, normal, duro | scenario |
| Valor esperado | Promedio pesado por la probabilidad | 240.000 € con 6 módulos | expected value |
| Error del promedio | El plan con el promedio no cuesta el promedio | promete 80.000 €, cuesta 280.000 € | flaw of averages |
| Primera etapa | Lo que decides antes de saber | comprar módulos | first-stage, here-and-now |
| Segunda etapa, recurso | Lo que corriges después de saber | urgencias, envíos | second-stage, recourse |
| No anticipación | Una decisión de hoy no depende del escenario | $q$ sin índice $s$ | non-anticipativity |
| Forma extendida | Una copia de la segunda etapa por escenario | $N$ transportes unidos por el stock | extensive form, deterministic equivalent |
| EVPI | Cuánto vale saber el futuro | 160.000 € | expected value of perfect information |
| VSS | Cuánto vale modelar la incertidumbre | 40.000 € | value of the stochastic solution |
| Vendedor de periódicos | Comprar mientras $P(D \ge k) \cdot$ urgencia > compra | 16 % | newsvendor, critical ratio |
| Monte Carlo | Sortear muchos escenarios y promediar | 200.000 inviernos | Monte Carlo simulation |
| SAA | Resolver la forma extendida con una muestra | 10 → 10.000 escenarios | sample average approximation |
| Fuera de la muestra | Evaluar el plan con escenarios nuevos | promete 160.000 €, cuesta 360.000 € | out-of-sample evaluation |
| Intervalo de confianza | $\bar{x} \pm 1{,}96\, s / \sqrt{N}$ | ancho de 4.000 € con 100.000 inviernos | confidence interval |
| Simulación de eventos | El reloj salta de evento en evento | muelles de Łeba | discrete-event simulation |
| Correlación | Los escenarios golpean a todos a la vez | tormentas en los 4 parques | correlated uncertainty |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $S$, $\pi_s$ | escenarios y su probabilidad, $\sum_s \pi_s = 1$ | 0,4 · 0,4 · 0,2 |
| $\mathbb{E}[\cdot] = \sum_s \pi_s (\cdot)_s$ | valor esperado | costo esperado |
| $q$ contra $e_s$ | sin índice $s$: primera etapa; con índice $s$: segunda | compra contra urgencias |
| $\mathbb{Z}_{\ge 0}$ | enteros no negativos | módulos |
| $D \sim \text{Poisson}(\lambda)$ | conteo aleatorio con media $\lambda$ | fallas del invierno |
| $\hat{\theta}$ | estimación con una muestra | el costo que promete el SAA |
| $\frac{1}{N} \sum_{s=1}^{N}$ | promedio de la muestra: cada escenario pesa $1/N$ | SAA |
| $\bar{x}$, $s$, $s / \sqrt{N}$ | media, desviación y error estándar | intervalo de confianza |
| $\text{EVPI} = \text{RP} - \text{WS}$ | valor de la información perfecta | 160.000 € |
| $\text{VSS} = \text{EEV} - \text{RP}$ | valor de la solución estocástica | 40.000 € |
| $x_{bfs}$, $D_{fs}$ | variable y dato con índice de escenario | envíos por invierno |

## 8. Ejercicios
1. **Urgencia más barata.** Un proveedor ofrece traer el módulo de urgencia por 150.000 € en vez de 250.000 €. Antes de resolver, usa la regla del módulo k
   con los 3 escenarios de la sección 2: ¿cuántos módulos compras ahora? Comprueba con `dos_etapas(ESC, urgencia=150_000)`.
2. **¿Vale la pena el cuarto muelle?** En la sección 4, las réplicas de 3 y 4 muelles usan las **mismas semillas** (`range(200)`).
   Calcula el intervalo de la **diferencia** pareada `replicas[3] - replicas[4]`. Repite con semillas distintas para 4 muelles (`range(1000, 1200)`).
   ¿Cuál intervalo es más angosto? ¿Por qué?
3. **Tormentas sin correlación.** Genera 1.000 escenarios con `escenarios_red(1000, semilla=1000, correlado=False)`: cada parque con su propio invierno.
   Resuelve. Antes, adivina: ¿guarda más o menos módulos? Evalúa ese plan en `EVAL` (que sí es correlado) con `costo_fuera`. ¿Cuánto se equivoca lo que promete?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"How do you handle uncertainty in an optimization model?"*
> *"First I check if it matters: I evaluate the deterministic plan with Monte Carlo or simulation, out of sample. Planning with averages is usually optimistic,
> because costs are asymmetric. In my spare parts example the average-based plan promised 80k and actually cost 280k in expectation.
> If uncertainty matters and some decisions can be corrected later, I use a two-stage stochastic model: first-stage decisions are fixed across scenarios,
> second-stage recourse variables are indexed by scenario. I solve it with sample average approximation, increase the number of scenarios until the plan is stable,
> and always evaluate on a fresh, larger set of scenarios. I report EVPI and VSS so the business sees what the stochastic model is worth."*

Seguimiento: *"How many scenarios do you need? And what if scenarios are correlated?"*
> *"Enough that the first-stage solution stops changing and the in-sample cost matches the out-of-sample cost. In my example that was around 200 to 1,000 scenarios.
> Correlation matters a lot: storms hit all wind farms at once. If you sample farms independently, the model under-stocks and its promised cost is too low.
> The scenario generator has to reproduce the joint behaviour, not just the marginals."*

Seguimiento: *"When would you use simulation instead of optimization?"*
> *"Simulation evaluates a design under full variability, like queues at a port base. Averages said two berths were enough at 83% utilization,
> but simulation showed 5.6 hours of average waiting. Optimization chooses; simulation checks. Together they form a simple digital twin."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
for g in [250_000, 150_000]:
    print(f"Urgencia {eur(g)}: regla P(fallas >= k) > {40_000 / g:.0%}")
    for k in range(1, 8):
        print(f"   módulo {k}: P(fallas >= {k}) = {ESC.loc[ESC['fallas'] >= k, 'prob'].sum():.1f}")
    me = dos_etapas(ESC, urgencia=g)
    print(f"   → compro {pyo.value(me.q):.0f}, costo esperado {eur(pyo.value(me.obj))}")

Con urgencia a 150.000 €, el módulo k vale la pena si $P(\text{fallas} \ge k) > 40.000 / 150.000 = 27\,\%$.
Los módulos 1 y 2 tienen 60 %: sí. Del 3 al 6 tienen 20 %: ya no. Se compran **2**, el promedio, por 200.000 € esperados.
El plan saltó de 6 a 2 por un cambio de precio. Por eso el precio de la urgencia es el dato que más hay que cuidar: pídelo con su rango (lección 2).

In [ ]:
# Ejercicio 2
dif_misma = replicas[3] - replicas[4]
otras = pd.Series([un_mes(4, s) for s in range(1000, 1200)])
dif_otra = replicas[3] - otras
for nombre, dif in [("mismas semillas", dif_misma), ("semillas distintas", dif_otra)]:
    media_d, bajo, alto = intervalo(dif)
    print(f"{nombre:>18}: diferencia {media_d:.2f} h, IC 95 % [{bajo:.2f}, {alto:.2f}], ancho {alto - bajo:.2f} h")

El cuarto muelle baja la espera en **0,51 h** por barco. Con las mismas semillas el intervalo mide 0,06 h; con semillas distintas, 0,08 h: **28 % más angosto**.
Por qué: cada réplica de 3 y de 4 muelles ve **los mismos barcos**, con las mismas llegadas y cargas. Un mes con muchas llegadas es malo para los dos, y esa parte se cancela al restar.
Solo queda la diferencia que causa el muelle extra. Se llama ***common random numbers*** (CRN). Con semillas distintas, el ruido de los dos meses se suma.
Para el mismo ancho sin CRN harían falta unas $(0{,}08 / 0{,}06)^2 \approx 2$ veces más réplicas. Con 2 contra 3 muelles la ganancia es menor: la diferencia es tan grande que se ve con cualquier método.

In [ ]:
# Ejercicio 3
mi = red_dos_etapas(escenarios_red(1000, semilla=1000, correlado=False))
print("Sin correlación: stock", stock_de(mi), "| promete", eur(pyo.value(mi.obj)))
print("Ese plan, evaluado con tormentas correladas:", eur(costo_fuera(tuple(stock_de(mi).items()))))
print("El plan correlado (1.000 escenarios):       ", escala.loc[escala["escenarios"] == 1000, "real"].item())

Sin correlación el modelo guarda **un módulo menos** (Ustka con 2) y promete unos 366.000 €. En el mundo con tormentas correladas, ese plan cuesta unos 450.000 €: la promesa queda 84.000 € abajo (19 %).
Por qué: si cada parque tiene su invierno, es raro que los 4 fallen mucho a la vez. Las fallas de unos se compensan con la calma de otros, y el stock compartido alcanza.
Con correlación, un invierno duro es duro en todos: la demanda total varía mucho más.
**Lección:** el generador de escenarios tiene que copiar cómo se mueven los datos **juntos**, no solo cada uno por separado.